# 10.2 图像识别实践 (Image Recognition in Practice)

## 📚 本章概览 (Overview)

**学习目标**：
- 掌握轻量模型在图像分类/检测/检索任务中的训练与评估全流程
- 理解数据增强策略对模型泛化能力的影响
- 学会构建以图搜图系统和评估检索质量
- 掌握模型校准方法，确保置信度可靠

**核心问题**：轻量模型在具体图像任务上如何达到可上线精度？训练过程中有哪些关键调优点？

🏢 **业务场景**：零售客户要求在货架图像中识别 200+ SKU（Stock Keeping Unit, 库存量单位），准确率 ≥ 95%，且能检测缺货和错放。医疗客户则需要从皮肤镜图像中区分良恶性病变，要求高召回（宁可多报、不可漏报）。两个场景对模型的诉求截然不同——一个是多分类 + 检测，一个是不平衡二分类 + 高召回。

**知识地图**：本章基于 10.1 的选型结果，深入图像任务的实战训练。是 10.3 视频理解的基础（图像是视频的单帧）。

**预计学习时间**：3-4 小时

## 🎯 动机与背景 (Motivation)

### 为什么图像识别需要专门的实践指导？

在 ImageNet 上 80% 的精度不等于在你的业务数据上 80% 的精度。从预训练模型到可上线模型之间，存在一条“最后一公里”的鸿沟：数据分布差异、类别不平衡、标注噪声、推理环境差异……

本章的目标就是填平这条鸿沟。

### 要解决的实际问题

1. 通用预训练模型在零售/医疗/安防场景精度不够，如何系统化提升？
2. 数据增强、学习率调度、损失函数——每个选择对最终精度的影响有多大？
3. 模型说“90% 置信度”，实际准确率到底是多少？

In [ ]:
# 🔬 Micro Practice 1: Lightweight image classification pipeline
# Goal: End-to-end training with TinyViT on a custom dataset

import torch
import torch.nn as nn
import timm
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# TODO: Set up data pipeline with albumentations
# TODO: Load TinyViT, replace classification head
# TODO: Train with proper LR schedule and evaluation

print("Classification pipeline setup")

In [ ]:
# 🔬 Micro Practice 2: Data augmentation strategy comparison
# Goal: Measure the impact of different augmentation strategies

import albumentations as A
from albumentations.pytorch import ToTensorV2

# TODO: Define 4 augmentation strategies (none, basic, AutoAugment, custom)
# TODO: Train same model with each strategy
# TODO: Compare validation accuracy and overfitting gap

print("Augmentation comparison setup")

In [ ]:
# 🔬 Micro Practice 3: Object detection with YOLO-nano
# Goal: Train a lightweight detector on custom data

# TODO: Set up YOLO-nano with custom class definitions
# TODO: Train and evaluate mAP@0.5, mAP@0.5:0.95
# TODO: Analyze false positives and false negatives

print("Object detection setup")

In [ ]:
# 🔬 Micro Practice 4: Detection error analysis
# Goal: Decompose mAP into component errors

# TODO: Implement error analysis by category, size, and aspect ratio
# TODO: Identify top-3 failure modes and propose fixes

print("Error analysis setup")

In [ ]:
# 🔬 Micro Practice 5: Image retrieval with FAISS
# Goal: Build an image-to-image search system

import faiss
import numpy as np

# TODO: Extract features from a pretrained model
# TODO: Build FAISS index and implement search
# TODO: Evaluate retrieval accuracy (Recall@K, mAP)

print("Image retrieval setup")

In [ ]:
# 🔬 Micro Practice 6: Multi-label classification
# Goal: Predict multiple attributes per image simultaneously

# TODO: Set up multi-label dataset (e.g., product attributes)
# TODO: Train with BCE loss and evaluate per-label AP

print("Multi-label classification setup")

In [ ]:
# 🔬 Micro Practice 7: Model calibration analysis
# Goal: Evaluate and improve confidence calibration

from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

# TODO: Plot reliability diagram
# TODO: Compute ECE (Expected Calibration Error)
# TODO: Apply temperature scaling and compare

print("Calibration analysis setup")

In [ ]:
# 🔬 Micro Practice 8: Comprehensive evaluation report
# Goal: Generate a production-ready evaluation report

# TODO: Combine all metrics into a structured report
# TODO: Include confusion matrix, per-class metrics, calibration curve
# TODO: Output actionable recommendations

print("Evaluation report setup")

## 📖 理论基础 (Theory)

### 3.1 图像分类的数学形式

给定图像 $x \in \mathbb{R}^{H \times W \times 3}$，模型 $f_\theta$ 输出类别概率分布：

$$P(y|x) = \text{softmax}(f_\theta(x))$$

训练目标是最小化交叉熵损失。但真实场景中，我们需要关注的不只是 accuracy——还有 per-class recall、校准误差、以及对分布偏移的鲁棒性。

### 3.2 mAP 的直观理解

mAP (mean Average Precision, 平均精度均值) 是检测任务的核心指标：
- Precision-Recall 曲线下面积
- 对每个类别分别计算 AP，取平均
- mAP@0.5 (IoU > 0.5) vs mAP@0.5:0.95（更严格）

### 3.3 模型校准

ECE (Expected Calibration Error, 期望校准误差) 衡量置信度与准确率的匹配程度。一个校准良好的模型，说“90% 置信度”的样本中，应该有约 90% 是正确的。

## 🔨 从零实现 (Implementation from Scratch)

### 数据增强管线的 NumPy 实现

理解常用增强操作背后的数学变换。

In [ ]:
# NumPy implementation of common augmentations
import numpy as np
from numpy.random import uniform, randint

def random_horizontal_flip(image, p=0.5):
    """
    Randomly flip image horizontally.
    
    Args:
        image: (H, W, C) numpy array
        p: probability of flipping
    
    Returns:
        flipped image
    """
    # TODO: Implement
    pass

def mixup(image1, image2, label1, label2, alpha=0.2):
    """
    MixUp augmentation: linearly interpolate two images and labels.
    
    Args:
        image1, image2: (H, W, C) numpy arrays
        label1, label2: one-hot label vectors
        alpha: Beta distribution parameter
    
    Returns:
        mixed_image, mixed_label
    """
    # TODO: Implement
    pass

print("Augmentation NumPy implementation")

In [ ]:
# NumPy implementation of mAP computation
def compute_ap(precision, recall):
    """
    Compute Average Precision from precision-recall curve.
    
    Args:
        precision: list of precision values at each recall level
        recall: list of recall values
    
    Returns:
        ap: average precision (area under PR curve)
    """
    # TODO: Implement 11-point interpolation or all-point interpolation
    pass

print("mAP computation NumPy implementation")

## ⚙️ 工程化实现 (Engineering Implementation)

### PyTorch 训练管线最佳实践

In [ ]:
# Production-grade training pipeline with mixed precision, gradient accumulation
from torch.cuda.amp import autocast, GradScaler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

class Trainer:
    """
    Production-grade classification trainer.
    
    Features:
    - Mixed precision training (AMP)
    - Gradient accumulation for large effective batch size
    - Cosine annealing with warm restarts
    - TensorBoard logging
    - Checkpoint management
    """
    pass

print("Training pipeline setup")

## 🚀 综合项目 (Capstone Project)

### 项目：多场景商品识别系统

**需求**：构建一个零售货架商品识别系统，包含分类和检测两个子任务。

**基础实现（必做）**：
1. 训练图像分类模型识别 50+ 商品类别
2. 训练目标检测模型定位商品位置
3. 集成分类+检测的级联管线
4. 输出完整的评估报告（混淆矩阵、mAP、ECE）

**进阶挑战（选做）**：
1. 实现以图搜图补货建议（输入缺货商品图 → 推荐替代品）
2. 多标签场景：同时识别商品品牌、口味、规格
3. 少样本扩展：用 10 张图注册新 SKU 而不重训全模型

In [ ]:
# 🚀 Capstone: Retail product recognition system
# TODO: Implement complete classification + detection pipeline

print("Capstone project setup")

## ❓ 常见问题与调试 (FAQ & Debugging)

### Q1: 训练集精度高验证集精度低？
经典过拟合。排查：数据增强是否充分、Dropout 比例、weight decay 强度、训练数据是否太少。

### Q2: 某些类别精度始终很低？
检查：类别样本数是否过少（考虑 oversampling 或 Focal Loss）、标注质量、类间相似度（考虑层次分类）。

### Q3: 检测模型小目标漏检严重？
小目标检测是通用难点。优化方向：提高输入分辨率、调整 anchor 尺寸、使用 FPN (Feature Pyramid Network, 特征金字塔网络) 多尺度特征。

### Q4: 模型校准后精度不变但置信度更可信？
这是正常的！温度缩放改变的是 softmax 的“锐度”（temperature），不影响 argmax 结果（分类决策不变），但让置信度数值更接近真实概率。

### Q5: 以图搜图搜出完全不相关的结果？
可能原因：用了分类模型的中间层特征（任务不匹配），应该用专门训练的 embedding 模型或 CLIP 视觉编码器。

## 📝 总结与展望 (Summary)

### 核心要点回顾
1. 数据增强是提升泛化能力的最廉价手段——不是调参，是调增强
2. mAP 诊断需要分解为 per-class / per-size 分析，不能只看总数
3. 模型校准是上线前的必要检查——用户看到的不是 argmax，是置信度分数
4. 以图搜图的关键是特征空间的语义对齐，而非像素级相似

### 与后续章节的联系
- **10.3 视频理解**：将单帧检测扩展为时序分析
- **10.4 边缘部署**：将训练好的图像模型量化部署

### 💡 思考题
1. 零售场景中，新增一种饮料口味（视觉上与已有款几乎一样），如何避免模型混淆？
2. 医疗影像中正负样本 1:100，Focal Loss 和重采样哪个更好？什么情况下两者一起用？
3. 你的模型在测试集上 95% 准确率，但上线第一天就被用户投诉“经常识别错”。这是什么问题？怎么在训练阶段发现？

### 下一步
进入 10.3 视频理解与监控，处理连续帧中的时序信息。